# 3번 문제 — 현재 작업 실행본

스냅샷 시각: **2026-10-10 06:14:41 UTC**. 최종 제출 확정본이 아니라, 이 시점의 작업 결과를 온라인 Jupyter에서 확인하기 위한 실행본입니다.

이 파일을 **해당 문항의 원본 Notebook과 같은 폴더**에 업로드하고 **Kernel → Restart & Run All**을 실행하세요.
기존 `data/` 폴더를 사용하며, 마지막 셀에서 현재 작업 폴더의 **`submission.csv`**를 생성합니다.
기존 `submission.csv`가 있으면 이 결과로 갱신합니다. 문항마다 별도 폴더에서 실행하세요.
데이터 위치가 다르면 첫 코드 셀의 `DATA_ROOT`만 수정하세요. 추가 산출물 파일 업로드나 패키지 설치는 필요 없습니다.

**실행 내용:** 현재 CPU 분류기 소스를 모두 포함했습니다. `train.csv`에서 주 증상별 문자 TF-IDF 검색기를 학습하고, 주 증상 규칙과 검색 결과로 `test.csv` 180건을 실제 재예측합니다. 내장된 기존 예측은 일치 여부를 확인하는 용도로만 사용합니다.

실행 대상은 사용자 제공 온라인 환경(Python 3.9.7, CPU 2개, RAM 4 GiB)입니다.
**온라인 동일 환경 검증 미완료:** 저장된 출력은 로컬 Python 3.12 실행 결과입니다.
Python 3.9 문법 검사와 로컬 전체 실행은 실제 온라인 커널 실행 확인과 구분합니다.
아래 원래 문제 지문 뒤에 실행 셀이 있습니다.

# 군 장비 정비 보고서 분석을 통한 고장 유형 분류하기

본 임무의 목적은 주어진 **군 장비의 정비 및 고장 보고서를 분석하여 고장 유형을 예측하는 분류기**를 만드는 것입니다. 보고서에는 장비에서 관찰된 증상과 점검 결과가 한국어 문장으로 기록되어 있습니다. 고장 유형은 총 6개로 구분되며, 모델은 각 보고서가 어떤 고장 유형에 해당하는지를 분류해야 합니다. 모든 고장 유형을 고르게 잘 분류하는 것이 중요하기 때문에 평가는 전체 정확도 대신 6개 고장 유형의 **Macro F1 점수**를 기준으로 진행합니다.


## 훈련 데이터

`train.csv`는 세 개의 필드로 구성되며 총 **540개의 레코드**가 제공됩니다.

| 필드 | 타입 | 설명 |
| --- | --- | --- |
| `id` | 문자열 | 각 레코드에 부여된 식별자입니다. |
| `report` | 문자열 | 군 장비의 상태, 증상 및 점검 결과가 기록된 한국어 정비 보고서입니다. |
| `fault_type` | 문자열 | 예측 대상이 되는 고장 유형입니다. |

`fault_type`은 다음 6개 중 하나의 값을 가집니다.

| `fault_type` | 의미 |
| --- | --- |
| `ENGINE` | 엔진 및 동력 계통 이상 |
| `ELECTRICAL` | 전기 및 전원 계통 이상 |
| `HYDRAULIC` | 유압 계통 이상 |
| `COMMUNICATION` | 통신 계통 이상 |
| `MOBILITY` | 주행, 조향 및 제동 계통 이상 |
| `COOLING` | 냉각 계통 이상 |

아래는 훈련 데이터의 일부 예시입니다.

| `report` | `fault_type` |
| --- | --- |
| 압력은 형성되나 기준값보다 낮아 무거운 부하에서 동작 범위가 제한됨. | `HYDRAULIC` |
| 근거리에서는 통신되나 거리가 늘어나면 신호가 급격히 약해짐. | `COMMUNICATION` |
| 타이어 편마모가 관찰되나 공기압과 주행 안정성은 기준 범위임. | `MOBILITY` |
| 연결 피팅에 습윤 흔적이 있으나 시간 경과 후에도 양이 증가하지 않음. | `HYDRAULIC` |
| 속도가 높아지면 차체 진동이 커져 저속으로만 운행 가능한 상태임. | `MOBILITY` |

일부 보고서에는 실제 고장과 직접 관련되지 않은 다른 계통의 점검 결과도 포함될 수 있습니다. 따라서 특정 단어 하나만을 기준으로 판단하기보다는 **보고서 전체의 증상과 문맥을 분석하여 고장 유형을 하나로 판단해야 합니다.**

이 데이터를 이용하여 고장 유형 분류 모델을 학습합니다.

## 테스트 데이터

`test.csv`에는 앞서 만든 모델로 분류해야 할 **180개의 레코드**가 있습니다. 테스트 데이터에는 정답인 `fault_type`이 제공되지 않습니다.

| 필드 | 타입 | 설명 |
| --- | --- | --- |
| `id` | 문자열 | 각 레코드의 식별자입니다. |
| `report` | 문자열 | 분류해야 할 군 장비 정비 및 고장 보고서입니다. |

아래는 테스트 데이터의 일부 예시입니다.

| `id` | `report` |
| --- | --- |
| `TST0001` | 핸들을 움직여도 진행 방향이 바뀌지 않아 조향 제어를 상실했다. 별도 실시한 전원 점검은 정상 범위였다. |
| `TST0002` | 첫 기동에 여러 번의 시도가 필요하지만 일단 가동되면 저부하 운전은 가능하다. |
| `TST0003` | 가벼운 부하는 처리되지만 큰 부하에서는 충분한 압력이 유지되지 않는다. |
| `TST0004` | 전원 투입 명령에도 표시부와 제어부가 모두 무응답이다. |
| `TST0005` | 송신을 시도하면 상대 장비에서 내용이 거의 수신되지 않는다. 배터리 전압은 정상 범위다. |

학습한 모델에 `report` 값을 입력하여 각 레코드의 고장 유형을 예측한 뒤, 아래에서 설명하는 제출 파일 형식에 맞게 결과를 만들어야 합니다.



## 제출 파일 포맷

`test.csv`의 각 레코드에 대해 고장 유형을 예측한 뒤 아래와 같은 형식의 CSV 파일로 저장합니다. 파일 이름은 `submission.csv`로 지정합니다.

결과의 `id` 값은 `test.csv`의 값을 그대로 사용해야 합니다. `fault_type`에는 앞서 `report`를 바탕으로 예측한 하나의 고장 유형을 사용합니다.

제출 결과는 `test.csv`의 `id` 순서와 동일하게 정렬해야 합니다.

`submission.csv`의 예시는 다음과 같습니다. 아래 값은 **파일 형식을 보여주기 위한 예시일 뿐 실제 정답과는 관계가 없습니다.**

| `id` | `fault_type` |
| --- | --- |
| `TST0001` | `MOBILITY` |
| `TST0002` | `ENGINE` |
| `TST0003` | `HYDRAULIC` |
| `TST0004` | `ELECTRICAL` |
| `TST0005` | `COMMUNICATION` |

- `id`: `test.csv`에서 제공된 ID 값을 그대로 사용합니다.
- `fault_type`: 모델이 예측한 고장 유형으로 다음 6개 값 중 하나여야 합니다.
  - `ENGINE`
  - `ELECTRICAL`
  - `HYDRAULIC`
  - `COMMUNICATION`
  - `MOBILITY`
  - `COOLING`

## 평가 기준

여섯 가지 고장 유형을 모두 고르게 잘 분류하는 것이 중요하기 때문에 **Macro F1 Score**를 주요 평가 지표로 사용합니다. 각 고장 유형별로 precision과 recall을 이용하여 F1 score를 계산한 뒤, 6개 클래스의 F1 score를 동일한 비중으로 평균하여 Macro F1을 계산합니다.

- **Precision**: 특정 고장 유형이라고 예측한 데이터 중 실제로 해당 고장 유형인 비율
- **Recall**: 실제 특정 고장 유형인 데이터 중 모델이 해당 유형으로 정확히 찾아낸 비율
- **F1 Score**: `2 × Precision × Recall / (Precision + Recall)`
- **Macro F1**: 6개 고장 유형 각각의 F1 Score를 계산한 뒤 단순 평균한 값

## 실행 1. 환경과 경로

In [1]:
import os
for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_variable] = "2"

import csv
import hashlib
import json
import platform
import sys
import time
from pathlib import Path
from importlib import metadata
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

# 보통 수정할 필요가 없습니다. 자동 탐색 실패 시 data 폴더 경로를 직접 지정하세요.
DATA_ROOT = None  # 예: Path("./data")
OUTPUT_PATH = Path.cwd() / "submission.csv"
started_at = time.perf_counter()
print("Python:", platform.python_version(), "|", sys.executable)
print("작업 폴더:", Path.cwd())
print("NumPy:", np.__version__, "| pandas:", pd.__version__)
for _package in ("torch", "scikit-learn"):
    try:
        print(_package + ":", metadata.version(_package))
    except metadata.PackageNotFoundError:
        print(_package + ": 미설치 (이 실행본에서 필수 아님)")
print("추가 패키지 설치나 GPU 연결 없이 실행합니다.")

Python: 3.12.3 | /config/workspace/.venv/bin/python
작업 폴더: /config/workspace/race90/notebook-checks/current/Problem #3
NumPy: 2.5.3 | pandas: 3.0.6
torch: 2.14.1+cpu
scikit-learn: 1.9.1
추가 패키지 설치나 GPU 연결 없이 실행합니다.


## 실행 2. 원본 데이터 및 기존 결과 확인

In [2]:
# 이 데이터는 현재 작업에서 얻은 예측·검토 산출물입니다. 모델 가중치나 정답 데이터가 아닙니다.
import base64
import zlib

_packed_snapshot = (
    'eNqVWU1vG1UU/SujWafRu+/7mVVxo2IpH6h1FhVCo/HMm2aEPWPNjAMhyi4LBDukSkGKUCu1OxYIAeqCX4Sd/8AztIi6IM6ToiSO'
    'j+/XmXvfyX2XadH5fPBllg/ZaijSUcoZ1/eIha+E6RHJkaTkdDpO91L/hS9WQ9022aItfYDe3Vxvvv8hGX98mqx/vV6/uvn99Y/r'
    'V78lQ5fXzX7Rn69f3CZ3z77bfP3L3bPbZP3ty2Tw/bB9J9k8v9789PMWvfnm5eb5bRIsbW6+2rx+ETzVzXI1ZAGW9Wc5VzodXaZ/'
    'Gw2OXe5zQZq8tFw4y6iUjFWFqrgvJZdFobkrhXNKMu+l4dqW2jnOCq+s0TMbXLwNJFgTpaJCV0o6xUqllOaarJl5WeW5VTPtZxUj'
    'WwrGlddspk0VLNHMUFFSzrRKr/bSzle+803hs2Xny7rYlqlPR59cpnUZXEwfTxljFPxW+Wo+ZMPFclvAo5MPJ4eT6ZNg4R0g3wUe'
    'HD+cHB/swsQu7KMnDx7dPz2cjHeR8j2Dhwfj6aPJ+P7hLlTtQscnR0enxwE6nZwc76I1FqmBI7WYQYcaJBaTDxFcKOI4FCaKJJQ+'
    '/QtJJ4eT44e7OA3iDIizIM5hOM7gGnICTfIYvrmA6s0l6Fzh+WhwGHCQG25x3w6GCoY+ugIkSHDYogAtguyIqNEmNF4kAz1FwoKU'
    'CxcTqIyacJLQ8suoTpLwjJMSRioYqaNixfiS4LSTeDspBj4CCtUKCj+GlEBtyphqKgVVU2mUTAVOPIVpBQVrBR3VSZog95qDVdco'
    'PVqiQIwZjU86baIqZKPQeBsZBiVm0CYyHHvijIhJyGBizoBizoBizhi8jhZtDeOgXCzDYrRY71iQF4vJOBs11yzIiwV5sQZ8Gi2q'
    'E6wDgQ49dhzGi4M1nAM1nAM1nMNGmovSAw7vGAdqAhcj4YhB84zQBQIxrG+IiagoJepeoUCN5W2iwkSHGjGwf4gwfogwGMdgWOsQ'
    'xUw1IlRTE8GygMhgGcHMEKrZiIOjjTjaPBxW1MQFahOSAsRhcnjMgCNu0DgtCoQJEljvCHjzRvAGgd7fIPy30aguilolkNBwwAYP'
    'GG4mAXMl4X0cSbSdJEeB4LyTEg8SPYikRoFoJ0mLR4ltSwldHhC6PCCFnURKYDB0sUNKgRnjR5CK0gjKggHArRO1OyCNTzsNTzsd'
    'peY0diRphUcadSZFrRJIw9MOXyOQieLMYOrO4PLB4MeTwf4/IhN1MBn4YDIGRuKDz+BU2f+j6tO9tPTnft4uF74ZsvN8Xpf59hZ2'
    'e3e8yIuuzSpKR7TP9tK8KFZdXly8edkv5/UQLKp7VTsvk37owger2pfJ065dLcPP2UXStN0i2PwyvFrkdZP0F4vl0C4+SHq/vXMO'
    '0TXtsI3rQfvn+91q7vukbvqlL4bwobwpk85XdRN+/7wezv66JK+bp4nvurbr95PpWR0+0Cd58o9EkuLMF5/tBfdDMFb6pQ/fwp+3'
    '19fJ2zz2g/eyrt7cQPfZ0GbLrm67bIt690KaXV39AQ9agow='
)
_snapshot_bytes = zlib.decompress(base64.b64decode(_packed_snapshot))
assert hashlib.sha256(_snapshot_bytes).hexdigest() == '5a717f53941d7320fb76cc7ee886aa8a971b8dd2de22d2addcc41f0984eec4a2'
snapshot = json.loads(_snapshot_bytes.decode('utf-8'))
del _packed_snapshot, _snapshot_bytes
print('산출물 스냅샷:', snapshot['created_at_utc'])
print('실행 방식:', snapshot['execution_mode'])

산출물 스냅샷: 2026-10-10 06:14:41 UTC
실행 방식: 현재 CPU 분류기를 train.csv로 학습한 뒤 test.csv 전체를 실제 재예측


In [3]:
QUESTION = 3

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def sha256_csv_text(path):
    # UTF-8 BOM과 줄바꿈 방식의 차이는 허용합니다.
    value = Path(path).read_bytes().decode("utf-8-sig").replace("\r\n", "\n")
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def data_candidates():
    if DATA_ROOT is not None:
        return [Path(DATA_ROOT).expanduser()]
    here = Path.cwd()
    return [here / "data", here,
            here / ("Problem #" + str(QUESTION)) / "data",
            here / ("q" + str(QUESTION)) / "data",
            here / "race90" / "competition" / ("q" + str(QUESTION)) / "data"]

def save_submission(frame):
    if frame.isna().any().any():
        raise ValueError("제출 결과에 결측값이 있습니다.")
    temporary = OUTPUT_PATH.with_name(OUTPUT_PATH.name + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8", lineterminator="\n")
    written = pd.read_csv(temporary, encoding="utf-8")
    pd.testing.assert_frame_equal(written, frame.reset_index(drop=True), check_dtype=False)
    temporary.replace(OUTPUT_PATH)
    print("저장 완료:", OUTPUT_PATH)
    print("행 수:", len(frame), "| 열:", list(frame.columns))
    print("SHA256:", sha256_file(OUTPUT_PATH))
    print("실행 시간: %.2f초" % (time.perf_counter() - started_at))
    display(frame.head(10))
    display(FileLink(str(OUTPUT_PATH.relative_to(Path.cwd()))))

REQUIRED_TRAIN = ['id', 'report', 'fault_type']
REQUIRED_TEST = ['id', 'report']
data_root = None
for candidate in data_candidates():
    if not all((candidate / name).is_file() for name in ("train.csv", "test.csv")):
        continue
    train_columns = pd.read_csv(candidate / "train.csv", nrows=0, encoding="utf-8-sig").columns
    test_columns = pd.read_csv(candidate / "test.csv", nrows=0, encoding="utf-8-sig").columns
    if set(REQUIRED_TRAIN).issubset(train_columns) and set(REQUIRED_TEST).issubset(test_columns):
        data_root = candidate.resolve()
        break
if data_root is None:
    raise FileNotFoundError("이 문항의 train.csv/test.csv를 찾지 못했습니다. 노트북을 해당 문제 폴더에 넣거나 DATA_ROOT를 수정하세요.")
for name, expected_hash in snapshot["input_csv_sha256"].items():
    if sha256_csv_text(data_root / name) != expected_hash:
        raise ValueError(name + " 내용이 현재 작업 스냅샷과 다릅니다. 올바른 문제 데이터인지 확인하세요.")
train = pd.read_csv(data_root / "train.csv", encoding="utf-8-sig")
test = pd.read_csv(data_root / "test.csv", encoding="utf-8-sig")
print("입력 확인:", data_root, "| train", train.shape, "| test", test.shape)

입력 확인: /config/workspace/race90/competition/q3/data | train (540, 3) | test (180, 2)


## 실행 3. 주 증상 추출과 CPU 분류기

다른 계통의 정상 점검 결과를 분리하여 주 증상을 추출합니다. 규칙으로 판별되지 않은 문장은 훈련 데이터의 주 증상과 문자 2~4-gram TF-IDF 유사도를 비교합니다.

In [4]:
import math
import re
from collections import Counter

def core_report(text):
    # Main observation follows one of the report's structural headers.
    headers = [r'운용자가\s*', r'정비반 보고:\s*', r'기능 시험 결과\s*',
               r'다음 사항을 확인함\.\s*',r'현장 정비 기록: .*?에서\s*',
               r'\[현상\]\s*',r'\[점검 메모\] .*?에서\s*',
               r'정비 메모 — .*?:\s*',r'운용자 인계(?:사항| 기록)\([^)]*\):\s*',
               r'(?:에 대한 )?재현 시험(?:에서| 결과,)\s*',
               r'점검 결과,\s*',r'운용 후 보고:\s*']
    found = []
    for pat in headers:
        m=re.search(pat,text)
        if m:found.append(m)
    if found:
        m=min(found,key=lambda x:x.start());text=text[m.end():]
    # A normal finding may be inserted immediately after the header.
    parts=text.split('.')
    while len(parts)>1 and re.search(r'가능성은 낮으며|특이사항이 없었다|정상으로 확인됐다|뒷받침하는 징후는 발견되지 않았다',parts[0]):
        parts=parts[1:]
    text='.'.join(parts).strip()
    endings = [r'이라고 보고함',r'15분 점검 중',r'정비 후 첫 운용에서',
               r'외기 온도는 정상 범위였음',r'증상은 2회 연속 재현됨',
               r'주변 환경 조건은 정상 범위였다',r'동일 현상이 재시험에서도',
               r'짧은 확인 운전에서도',r'동일 조건에서 두 차례',
               r'정지 후 다시 시작하면',r'\[대상\]',r'\[기타\]']
    text=re.split('|'.join(endings),text)[0]
    text=text.split('.')[0]
    return text.strip(' .')

def semantic_rule(text):
    """Domain-level component rules applied only to the primary observation."""
    s=core_report(text)
    if re.search(r'통신|송신|수신|송수신|신호|무선|안테나|네트워크|호출|채널|헤드셋|음성|통화|단말|메시지|자료 교환|데이터|외부 장비와의 연결',s):
        return 'COMMUNICATION'
    cooling = re.search(r'냉각|라디에이터|온도|과열|팬|풍량|뜨거|열 축적|열 관련|흡입구|공기 흡입부|외기 조건|내부 측정값|보조 탱크|보조 저장조|장시간.*보호 경보|연속 운전.*보호 경보',s)
    electrical_heat = re.search(r'(전원 케이블|전력선).*과열',s)
    if cooling and not electrical_heat:
        return 'COOLING'
    if re.search(r'바퀴|차륜|조향|브레이크|제동|타이어|휠|트레드|직진|방향 전환|차체|속도.*진동.*저속',s):
        return 'MOBILITY'
    if re.search(r'엔진|동력부|공회전|크랭킹|연소|점화|윤활|배기|회전수|가속.*회전|진동감.*가속|첫 기동|기계음|금속성 충격|노킹|회전이 급격히|부하 증가.*힘이 빠|오르막.*출력 부족',s):
        return 'ENGINE'
    if re.search(r'배터리|전압|전류|충전|축전|배선|전력|전기|발전|회로|계기|커넥터|단자|전원|표시등|표시 램프',s):
        return 'ELECTRICAL'
    if re.search(r'압력|실린더|펌프|피팅|작동유|유압|액추에이터|저압|연결부.*젖|저장 탱크|구동.*둔',s):
        return 'HYDRAULIC'
    return None

def grams(text):
    text=re.sub(r'\s+',' ',text)
    return Counter(text[i:i+n] for n in (2,3,4) for i in range(len(text)-n+1))

class ReportClassifier:
    def fit(self,reports,labels):
        unique={}
        for text,label in zip(reports,labels):
            core=core_report(text)
            if core in unique and unique[core]!=label:raise ValueError('Conflicting symptom labels')
            unique[core]=label
        self.cores=list(unique);self.labels=[unique[c] for c in self.cores]
        docs=[grams(c) for c in self.cores];df=Counter()
        for d in docs:df.update(d.keys())
        self.idf={g:math.log((1+len(docs))/(1+n))+1 for g,n in df.items()}
        self.vectors=[self.vector(d) for d in docs]
        return self
    def vector(self,counts):
        x={g:(1+math.log(n))*self.idf[g] for g,n in counts.items() if g in self.idf}
        norm=math.sqrt(sum(v*v for v in x.values())) or 1
        return {g:v/norm for g,v in x.items()}
    def predict_one(self,report):
        rule=semantic_rule(report)
        if rule:return rule
        x=self.vector(grams(core_report(report)))
        sims=[sum(v*vec.get(g,0) for g,v in x.items()) for vec in self.vectors]
        return self.labels[max(range(len(sims)),key=sims.__getitem__)]
    def predict(self,reports):return [self.predict_one(s) for s in reports]

## 실행 4. 학습 및 180건 예측

In [5]:
allowed = {"ENGINE", "ELECTRICAL", "HYDRAULIC", "COMMUNICATION", "MOBILITY", "COOLING"}
if len(train) != 540 or len(test) != 180:
    raise ValueError("훈련/테스트 행 수 불일치")
if not train["id"].is_unique or not test["id"].is_unique:
    raise ValueError("입력 ID가 중복됩니다.")
if train[["report", "fault_type"]].isna().any().any() or test["report"].isna().any():
    raise ValueError("입력 텍스트 또는 레이블 결측")
if set(train["fault_type"]) != allowed:
    raise ValueError("훈련 레이블 오류")
model = ReportClassifier().fit(train["report"].tolist(), train["fault_type"].tolist())
predictions = model.predict(test["report"].tolist())
df = pd.DataFrame({"id": test["id"].tolist(), "fault_type": predictions})
expected = pd.DataFrame(snapshot["reference_predictions"])
pd.testing.assert_frame_equal(df, expected)
print("학습 완료: 훈련 보고서", len(train), "개 / 고유 주 증상", len(model.cores), "개")
print("실제 CPU 재예측 180건이 현재 작업 결과와 모두 일치")
print("기존 개발 점검 Macro F1:", snapshot["development_validation"]["macro_f1"])
print("위 지표는 훈련 오류를 보고 규칙을 개선한 개발 점검 값이며, 독립 검증 또는 테스트 점수가 아닙니다.")
display(df["fault_type"].value_counts().rename("예측 수"))

학습 완료: 훈련 보고서 540 개 / 고유 주 증상 90 개
실제 CPU 재예측 180건이 현재 작업 결과와 모두 일치
기존 개발 점검 Macro F1: 1.0
위 지표는 훈련 오류를 보고 규칙을 개선한 개발 점검 값이며, 독립 검증 또는 테스트 점수가 아닙니다.


fault_type
MOBILITY         30
ENGINE           30
HYDRAULIC        30
ELECTRICAL       30
COMMUNICATION    30
COOLING          30
Name: 예측 수, dtype: int64

## 실행 5. 제출 CSV 저장

In [6]:
assert list(df.columns) == ["id", "fault_type"]
assert len(df) == 180 and df["id"].is_unique
assert df["id"].tolist() == test["id"].tolist()
assert set(df["fault_type"]).issubset(allowed)
save_submission(df)

저장 완료: /config/workspace/race90/notebook-checks/current/Problem #3/submission.csv
행 수: 180 | 열: ['id', 'fault_type']
SHA256: 6a9ae5b264a84a5fb851792ef9f29f25fb59db0135556799b5ae637fda4ed98a
실행 시간: 0.12초


,id,fault_type
0,TST0001,MOBILITY
1,TST0002,ENGINE
2,TST0003,HYDRAULIC
3,TST0004,ELECTRICAL
4,TST0005,COMMUNICATION
5,TST0006,ENGINE
6,TST0007,HYDRAULIC
7,TST0008,ENGINE
8,TST0009,HYDRAULIC
9,TST0010,COMMUNICATION


/config/workspace/race90/notebook-checks/current/Problem #3/submission.csv